<div align="center">
  <h2>
     <a href="https://github.com/rihab-l/Techniques-IA-TPs.git">
    Accéder au Notebook sur GitHub
    </a>
  </h2>
</div>

# Introduction

Dans ce TP, nous allons explorer les techniques de prétraitement des données 
et construire des pipelines en utilisant la bibliothèque scikit-learn.

Le preprocessing est une étape essentielle en Machine Learning, car :
- Il améliore la qualité des données
- Il permet d’éviter les biais
- Il rend les modèles plus performants

Nous allons appliquer ces techniques sur un dataset réel.

# 1. Objectifs

- Comprendre le preprocessing des données
- Encoder les variables catégorielles
- Normaliser les variables numériques
- Construire un pipeline complet
- Comparer plusieurs modèles

# 2. Import des librairies

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    PolynomialFeatures
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 3. Chargement des données

In [5]:
# Dataset exemple
df = sns.load_dataset("diamonds")

df.head()

,carat,cut,color,clarity,depth,table,price,x,y,z
0,0.23,Ideal,E,SI2,61.5,55.0,326,3.95,3.98,2.43
1,0.21,Premium,E,SI1,59.8,61.0,326,3.89,3.84,2.31
2,0.23,Good,E,VS1,56.9,65.0,327,4.05,4.07,2.31
3,0.29,Premium,I,VS2,62.4,58.0,334,4.20,4.23,2.63
4,0.31,Good,J,SI2,63.3,58.0,335,4.34,4.35,2.75


# 4. Exploration des données

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53940 entries, 0 to 53939
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype   
---  ------   --------------  -----   
 0   carat    53940 non-null  float64 
 1   cut      53940 non-null  category
 2   color    53940 non-null  category
 3   clarity  53940 non-null  category
 4   depth    53940 non-null  float64 
 5   table    53940 non-null  float64 
 6   price    53940 non-null  int64   
 7   x        53940 non-null  float64 
 8   y        53940 non-null  float64 
 9   z        53940 non-null  float64 
dtypes: category(3), float64(6), int64(1)
memory usage: 3.0 MB


In [10]:
df.describe()

,carat,depth,table,price,x,y,z
count,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000,53940.000000
mean,0.797940,61.749405,57.457184,3932.799722,5.731157,5.734526,3.538734
std,0.474011,1.432621,2.234491,3989.439738,1.121761,1.142135,0.705699
min,0.200000,43.000000,43.000000,326.000000,0.000000,0.000000,0.000000
25%,0.400000,61.000000,56.000000,950.000000,4.710000,4.720000,2.910000
50%,0.700000,61.800000,57.000000,2401.000000,5.700000,5.710000,3.530000
75%,1.040000,62.500000,59.000000,5324.250000,6.540000,6.540000,4.040000
max,5.010000,79.000000,95.000000,18823.000000,10.740000,58.900000,31.800000


In [12]:
# Vérifier les valeurs manquantes
df.isnull().sum()

carat      0
cut        0
color      0
clarity    0
depth      0
table      0
price      0
x          0
y          0
z          0
dtype: int64

# 5. Train / Test Split

In [15]:
train_set, test_set = train_test_split(df, test_size=0.2, random_state=42)

print("Train:", train_set.shape)
print("Test:", test_set.shape)

Train: (43152, 10)
Test: (10788, 10)


# 6. Séparation X / y

In [18]:
X_train = train_set.drop("price", axis=1)
y_train = train_set["price"]

X_test = test_set.drop("price", axis=1)
y_test = test_set["price"]

# 7. Identification des variables

In [21]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

print("Numériques:", num_cols)
print("Catégorielles:", cat_cols)

Numériques: Index(['carat', 'depth', 'table', 'x', 'y', 'z'], dtype='object')
Catégorielles: Index(['cut', 'color', 'clarity'], dtype='object')


# 8. Identification des types de variables

##  Identification des variables numériques et catégorielles

In [25]:
num_cols = X_train.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X_train.select_dtypes(include=["object", "category"]).columns

print("Variables numériques :", list(num_cols))
print("Variables catégorielles :", list(cat_cols))

Variables numériques : ['carat', 'depth', 'table', 'x', 'y', 'z']
Variables catégorielles : ['cut', 'color', 'clarity']


# 9. Construction du preprocessing

##  Pipeline de preprocessing

In [29]:
# Pipeline numérique
num_pipeline = Pipeline([
    ("scaler", StandardScaler())
])

# Pipeline catégoriel
cat_pipeline = Pipeline([
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Combinaison
preprocessor = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# 10. Pipeline complet

##  Pipeline final (prétraitement + modèle)

In [45]:
df = df.sample(3000, random_state=42)
pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(
        solver="liblinear",  
        max_iter=200
    ))
])

## 11. Entraînement

##  Entraînement du modèle

In [47]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('scaler',
                                                                   StandardScaler())]),
                                                  Index(['carat', 'depth', 'table', 'x', 'y', 'z'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['cut', 'color', 'clarity'], dtype='object'))])),
                ('model',
                 LogisticRegression(max_iter=200, solver='liblinear'))])

# 12. Évaluation

##  Évaluation des performances

In [49]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = pipeline.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("\nRapport :\n", classification_report(y_test, y_pred))

Accuracy : 0.9779384501297739

Rapport :
               precision    recall  f1-score   support

           0       0.98      0.98      0.98      5396
           1       0.98      0.98      0.98      5392

    accuracy                           0.98     10788
   macro avg       0.98      0.98      0.98     10788
weighted avg       0.98      0.98      0.98     10788



# 13. Pipeline avec Polynomial Features

##  Ajout de caractéristiques polynomiales

In [51]:
pipeline_poly = Pipeline([
    ("preprocessing", preprocessor),
    ("poly", PolynomialFeatures(degree=2)),
    ("model", LogisticRegression(max_iter=2000))
])

pipeline_poly.fit(X_train, y_train)

y_pred_poly = pipeline_poly.predict(X_test)

print("Accuracy (Polynomial) :", accuracy_score(y_test, y_pred_poly))

Accuracy (Polynomial) : 0.978494623655914


# 14. Comparaison L1 vs L2

##  Comparaison Régularisation L1 vs L2

In [53]:
# L1
pipeline_l1 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(penalty="l1", solver="liblinear"))
])

pipeline_l1.fit(X_train, y_train)
y_pred_l1 = pipeline_l1.predict(X_test)

# L2
pipeline_l2 = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(penalty="l2"))
])

pipeline_l2.fit(X_train, y_train)
y_pred_l2 = pipeline_l2.predict(X_test)

print("Accuracy L1 :", accuracy_score(y_test, y_pred_l1))
print("Accuracy L2 :", accuracy_score(y_test, y_pred_l2))

Accuracy L1 : 0.9784019280682239
Accuracy L2 : 0.9783092324805339


##  Conclusion

- Le pipeline permet d’automatiser le preprocessing et le modèle
- OneHotEncoder gère les variables catégorielles
- StandardScaler améliore la performance du modèle
- L1 permet la sélection de variables
- L2 stabilise le modèle

 Le pipeline est une pratique essentielle en Machine Learning professionnel